In [1]:
from src.DataSplitting import remove_daylightsaving
import numpy as np
import pandas as pd

In [2]:
ts = remove_daylightsaving(pd.read_hdf(r"C:\Users\Samuel\FH_Aachen\MasterThesis\Master_Thesis_EWI_2026\data\split\train\run_000259.h5", key="timeseries"))

In [3]:
cols = ["Electricity Load", "Hot Water Load", "Heating Load"]
hourly = ts[cols].resample("1min").mean()
print(hourly.shape, hourly.index.tz, hourly.isna().sum().to_dict())

(525600, 3) None {'Electricity Load': 0, 'Hot Water Load': 0, 'Heating Load': 0}


In [4]:
hourly.index = hourly.index.tz_localize(None)

In [5]:
import pandas as pd
run_id = "run_000259"  # whichever you loaded

long = pd.concat([
    pd.DataFrame({"unique_id": f"{run_id}|{c}|year",
                  "ds": hourly.index,
                  "y": hourly[c].to_numpy(dtype=float)})
    for c in cols
], ignore_index=True)

In [ ]:
[acf_features, stl_features, entropy, lumpiness,
                             stability, flat_spots, crossing_points, hurst]

In [7]:
from tsfeatures import tsfeatures, acf_features, stl_features, entropy, lumpiness, stability, flat_spots, crossing_points, hurst
#
# feats = tsfeatures(long, freq=1440, threads=1,
#                    features=[acf_features, stl_features, entropy, lumpiness,
#                              stability, flat_spots, crossing_points, hurst]) #
# feats.T

In [8]:
w = hourly.iloc[100*24:107*24]          # 7 days
long_w = pd.concat([
    pd.DataFrame({"unique_id": f"{run_id}|{c}|w7d", "ds": w.index,
                  "y": w[c].to_numpy(dtype=float)}) for c in cols
], ignore_index=True)
tsfeatures(long_w, freq=24, threads=1, features=[acf_features, stl_features, entropy, lumpiness,
                             stability, flat_spots, crossing_points, hurst]).T

,0,1,2
unique_id,run_000259|Electricity Load|w7d,run_000259|Heating Load|w7d,run_000259|Hot Water Load|w7d
hurst,0.773954,0.994678,NaN
crossing_points,16,2,30
flat_spots,12,32,58
stability,0.570786,1.04693,0.330467
lumpiness,0.076141,0.032515,1.308878
entropy,0.622845,0.109242,0.786544
nperiods,1,1,1
seasonal_period,24,24,24
trend,0.658198,0.994661,0.392806


In [10]:
import time
for f in [entropy, lumpiness, stability, flat_spots, crossing_points, hurst]:
    t = time.time()
    tsfeatures(long_w, freq=1440, threads=15, features=[f])
    print(f.__name__, round(time.time() - t, 1))

entropy 8.9
lumpiness 8.2
stability 7.9
flat_spots 7.9
crossing_points 8.0
hurst 7.9
